In [1]:
!nvidia-smi

Fri May 29 15:31:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.74                 Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   44C    P0              9W /   80W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 19.9 MB/s eta 0:02:03
     ---------------------------------------- 0.0/2.4 GB 23.1 MB/s eta 0:01:46
     ---------------------------------------- 0.0/2.4 GB 25.4 MB/s eta 0:01:36
     ---------------------------------------- 0.0/2.4 GB 25.5 MB/s eta 0:01:36
     ---------------------------------------- 0.0/2.4 GB 28.2 MB/s eta 0:01:26
      --------------------------------------- 0.0/2.4 GB 30.0 MB/s eta 0:01:21
      --------------------------------------- 0.0/2.4 GB 29.8 MB/s eta 0:01:21
      --------------------------------------- 0.0/2.4 GB 30.6 MB/s eta 0:01:19
      --------------------------------------- 0.1/2.4 GB 30.5 MB/s eta 0:01:19
      --------------------------------------- 0.1/2.4 GB 30.7 MB/s eta 0:01:18
     - -------------------------------------- 0.1/2.4 GB 30.1 MB/s eta 0:01:20
 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import torch
print(torch.cuda.is_available())

True


In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.5.1+cu121
12.1
True


In [4]:
import torch
import time 

device = "cuda" #choose GPU 

#take 2 random 5k*5k matrix to test computation speed of multiplier cirucit 
x = torch.randn(5000, 5000).to(device) 
y = torch.randn(5000, 5000).to(device)

start = time.time()
z = x @ y #@ means matrix multiplication

torch.cuda.synchronize() #CPU must WAIT until GPU finishes all pending operation so as to get accurate timing 

print("Time:", time.time() - start)

Time: 0.06269264221191406


In [5]:
import torch
import torch.nn as nn  #imports neural network module 
from torch.nn import functional as F
import urllib.request



batch_size = 32      # How many independent sequences to process in parallel
block_size = 8       # Maximum context length for predictions
max_iters = 3000     # Training iterations
eval_interval = 300  # How often to check validation loss
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 32          # Embedding dimension
n_head = 4           # Number of attention heads
n_layer = 3          # Number of transformer blocks

print(f"--- Running on device: {device} ---")


url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read() #text='Hello'

chars = sorted(list(set(text))) #text=['e','h','l','o']
vocab_size = len(chars)#4 
stoi = { ch:i for i,ch in enumerate(chars) }#{'a':0}
itos = { i:ch for i,ch in enumerate(chars) }#{0:'a'}
encode = lambda s: [stoi[c] for c in s] 
#converts 'cab' to integer using stoi 
decode = lambda l: ''.join([itos[i] for i in l])
#reverses the conversion 

data = torch.tensor(encode(text), dtype=torch.long) 
#data contains the 1D tensor with 5 elements tensor([1,0,2,2,3])
n = int(0.9 * len(data))#90% in training and 10% in testing 
train_data, val_data = data[:n], data[n:]

#The main aim of this get_batch func is to divide the data into x and y 
#where x is the input and y is the target (y is one step ahead)
# x=tran y=rans ,we make a dataset like this for training 

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    #we need to divide x and y in both train and val_data 
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    #suppose we have block_s=4 and batch_s=2+len=11
    #torch.randint(7,(2,)) ->  choose 2 starting points between 0 and 6
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    #torch.stack combines tensor
    #torch.stack  ->tran
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    #go ahead 1 index for rans 
    return x.to(device), y.to(device)

#Estimate if the model is improving 
@torch.no_grad() #disable gradient tracking 
def estimate_loss():
    out = {}
    model.eval() #puts the model in evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(200) #200 0 in tensor 
        for k in range(200):
            X, Y = get_batch(split) #for every random batch we generate a x and  y 
            logits, loss = model(X, Y) #logits is the raw performance score (in tensor)   
            #loss is pred-actual   
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() #switch back to training mode 
    return out


class Head(nn.Module):
    """ One head of self-attention """
    #n_embed is the number of values (vector) that defines each token(char) 
    #head_size=8 means that these 32 values are compressed to 8-dimensional attention vector 
    #each q,k,v layers compress it to 8 values (because multiple attentionheads are there that will learn different parts of the info) 
    #ach head learns only PART of the information.
    #n_head*head_size=n_embed (learning everything 
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False) #What information do I contain?
        self.query = nn.Linear(n_embd, head_size, bias=False)#What am I searching for?
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        #traingluar lower is used to mask models to see the future values of what they are trying to predict 
        #these are not trainable andd is hence saved as register_buffer()

    def forward(self, x):
        B, T, C = x.shape #Batch size,sequence length(block wise),embedding dimension 
        k, q = self.key(x), self.query(x) #matching keys to queries 
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) #multiply matrices and normalize by divide by c 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #trianglular lower intro for masking future values 
        wei = F.softmax(wei, dim=-1) #softmax function to convert score ->prob
        return wei @ self.value(x) #return 

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        #create multiple heads (4) with dimension 8 
        self.proj = nn.Linear(n_embd, n_embd) #input 32 and output 32 

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        #concatenate the head vector values according to last dimension (C) 
        return self.proj(out)

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential( #nn.Sequential in PyTorch is a container that lets you stack layers in order.
            nn.Linear(n_embd, 4 * n_embd),#Expansion
            nn.ReLU(), #introduce non-Linearity 
            nn.Linear(4 * n_embd, n_embd),#compression 
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x): #Residual connection workflow 
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #initializing embedding table with tokenID and dimensions 
        self.position_embedding_table = nn.Embedding(block_size, n_embd) #positional encoding 
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) #stack multiple transformer blocks ,* is for unpacking the list 
        self.ln_f = nn.LayerNorm(n_embd) 
        self.lm_head = nn.Linear(n_embd, vocab_size)#final layer called large language modelling head 
        #converts the dimension (32) to 65 scores for each character 

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        #creates a position for each embedding 
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None: #we we are predicting from scratch 
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #B*T is the total no. of preds
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #we need to flatten for cross entropy to work 
            #loss=-log(P(correct ans))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = TinyGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


print("--- Starting Training ---")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Save the model weights
torch.save(model.state_dict(), 'tiny_gpt_fp32.pth')
print("--- Training Complete! Model saved as 'tiny_gpt_fp32.pth' ---")

# Let's see what it generates!
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- Output Generation ---")
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

--- Running on device: cuda ---
--- Starting Training ---
step 0: train loss 4.2658, val loss 4.2689
step 300: train loss 2.5261, val loss 2.5194
step 600: train loss 2.3569, val loss 2.3747
step 900: train loss 2.2854, val loss 2.2806
step 1200: train loss 2.2072, val loss 2.2367
step 1500: train loss 2.1686, val loss 2.1986
step 1800: train loss 2.1371, val loss 2.1731
step 2100: train loss 2.1026, val loss 2.1560
step 2400: train loss 2.0836, val loss 2.1337
step 2700: train loss 2.0622, val loss 2.1397
step 2999: train loss 2.0513, val loss 2.1253
--- Training Complete! Model saved as 'tiny_gpt_fp32.pth' ---

--- Output Generation ---

And so fOsine this that Es whit dife me for on Romm thad as here seath thit wous aume eseechope oll,
And For mast loals is my eann geat
Prost I Sast graight in fatherpled,
Astiom is your vy whell is l


In [6]:
import time
import json
import os
import torch

print("--- Calculating Baseline Metrics ---")

# 1. Model Size & Parameter Count
total_params = sum(p.numel() for p in model.parameters())
file_size_mb = os.path.getsize('tiny_gpt_fp32.pth') / (1024 * 1024)

# Put model in evaluation mode
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# 2. Peak VRAM Usage
# We reset the memory tracker, do a small generation, and measure the peak
torch.cuda.reset_peak_memory_stats(device)
_ = model.generate(context, max_new_tokens=10)
peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)

# 3. Inference Speed (Tokens Per Second)
# WARMUP PASS: Wake up the GPU (Remember the "CUDA Warmup" effect!)
print("Warming up GPU...")
for _ in range(2):
    _ = model.generate(context, max_new_tokens=50)
torch.cuda.synchronize()

# ACTUAL BENCHMARK
print("Running speed benchmark...")
tokens_to_generate = 500
start_time = time.time()

# Generate 500 tokens
_ = model.generate(context, max_new_tokens=tokens_to_generate)
torch.cuda.synchronize() # Wait for GPU to finish

time_taken = time.time() - start_time
tokens_per_second = tokens_to_generate / time_taken

# 4. Save and Display Metrics
baseline_metrics = {
    "model_type": "Standard_FP32",
    "total_parameters": total_params,
    "file_size_mb": round(file_size_mb, 2),
    "peak_vram_mb": round(peak_memory_mb, 2),
    "generation_time_seconds": round(time_taken, 2),
    "tokens_per_second": round(tokens_per_second, 2)
}

print("\n=== FP32 BASELINE RESULTS ===")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

# Save to a JSON file so you have it for Phase 5
with open("fp32_baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=4)
    
print("\nMetrics successfully saved to 'fp32_baseline_metrics.json'!")

--- Calculating Baseline Metrics ---
Warming up GPU...
Running speed benchmark...

=== FP32 BASELINE RESULTS ===
model_type: Standard_FP32
total_parameters: 42369
file_size_mb: 0.19
peak_vram_mb: 306.14
generation_time_seconds: 2.46
tokens_per_second: 203.02

Metrics successfully saved to 'fp32_baseline_metrics.json'!
